<a href="https://colab.research.google.com/github/he380801-sketch/Prueba/blob/main/Random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import randint

# 1. Cargar datos
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Analisis /IA/2do parcial/DM.csv')

# 2. Preparar datos
X = df.drop(['Grupo', 'Participante'], axis=1)
y = df['Grupo']

# 3. Codificar variable objetivo
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 4. Dividir datos
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print("="*60)
print("RANDOM FOREST CLASSIFIER")
print("="*60)
print(f"Dimensiones de entrenamiento: {X_train.shape}")
print(f"Dimensiones de prueba: {X_test.shape}")
print(f"Número de clases: {len(label_encoder.classes_)}")
print(f"Clases: {label_encoder.classes_}")

Mounted at /content/drive
RANDOM FOREST CLASSIFIER
Dimensiones de entrenamiento: (720, 23)
Dimensiones de prueba: (180, 23)
Número de clases: 2
Clases: [0 1]


In [ ]:
# 5. Random Forest básico
print("\n" + "="*60)
print("VERSIÓN BÁSICA DE RANDOM FOREST")
print("="*60)

rf_basic = RandomForestClassifier(
    n_estimators=100,      # Número de árboles
    random_state=42,       # Reproducibilidad
    n_jobs=-1              # Usar todos los procesadores
)

rf_basic.fit(X_train, y_train)

# Predicciones
y_pred_basic = rf_basic.predict(X_test)
y_pred_proba_basic = rf_basic.predict_proba(X_test)

# Evaluación básica
print(f"Accuracy en entrenamiento: {rf_basic.score(X_train, y_train):.4f}")
print(f"Accuracy en prueba: {accuracy_score(y_test, y_pred_basic):.4f}")
print("\nClassification Report (básico):")
print(classification_report(y_test, y_pred_basic, target_names=[str(c) for c in label_encoder.classes_]))


VERSIÓN BÁSICA DE RANDOM FOREST
Accuracy en entrenamiento: 1.0000
Accuracy en prueba: 0.9000

Classification Report (básico):
              precision    recall  f1-score   support

           0       0.91      0.91      0.91       100
           1       0.89      0.89      0.89        80

    accuracy                           0.90       180
   macro avg       0.90      0.90      0.90       180
weighted avg       0.90      0.90      0.90       180



In [ ]:
# 6. Optimización con GridSearchCV (exhaustiva pero lenta)
print("\n" + "="*60)
print("OPTIMIZACIÓN DE RANDOM FOREST")
print("="*60)

# Opción 1: Grid Search (más precisa pero más lenta)
param_grid_rf = {
    'n_estimators': [100, 200, 300],           # Número de árboles
    'max_depth': [10, 15, 20, None],           # Profundidad máxima
    'min_samples_split': [2, 5, 10],           # Mínimo muestras para dividir
    'min_samples_leaf': [1, 2, 4],             # Mínimo muestras en hoja
    'max_features': ['sqrt', 'log2', None]     # Features por árbol
}

# Usar RandomizedSearchCV (más rápida para muchos parámetros)
print("Buscando mejores parámetros (esto puede tomar unos minutos)...")
random_search_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_grid_rf,
    n_iter=30,              # Probar 30 combinaciones aleatorias
    cv=5,                   # Validación cruzada de 5 pliegues
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search_rf.fit(X_train, y_train)

# Mejores parámetros
print(f"\nMejores parámetros: {random_search_rf.best_params_}")
print(f"Mejor accuracy CV: {random_search_rf.best_score_:.4f}")

# Mejor modelo
best_rf = random_search_rf.best_estimator_
y_pred_best = best_rf.predict(X_test)
y_pred_proba = best_rf.predict_proba(X_test)

print(f"Accuracy en prueba con mejor modelo: {accuracy_score(y_test, y_pred_best):.4f}")
print("\nClassification Report (mejor modelo):")
print(classification_report(y_test, y_pred_best, target_names=[str(c) for c in label_encoder.classes_]))


OPTIMIZACIÓN DE RANDOM FOREST
Buscando mejores parámetros (esto puede tomar unos minutos)...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Mejores parámetros: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 15}
Mejor accuracy CV: 0.9069
Accuracy en prueba con mejor modelo: 0.9111

Classification Report (mejor modelo):
              precision    recall  f1-score   support

           0       0.93      0.91      0.92       100
           1       0.89      0.91      0.90        80

    accuracy                           0.91       180
   macro avg       0.91      0.91      0.91       180
weighted avg       0.91      0.91      0.91       180

